# Day 6 — Spark SQL

**What you will learn:**
- Why SQL is a first-class citizen in Spark
- `createOrReplaceTempView()` vs `createOrReplaceGlobalTempView()`
- Running SQL with `spark.sql()`
- Mixing SQL results with the DataFrame API
- Managing views with `spark.catalog`

**Exam relevance:** Spark SQL (20%) — a dedicated exam domain, equally weighted with Architecture.

## 1. Why Spark SQL?

Spark SQL lets you query DataFrames using standard SQL. Both approaches produce **identical execution plans** — Catalyst optimises them the same way.

| | DataFrame API | Spark SQL |
|---|---|---|
| Style | Programmatic (method chaining) | Declarative (SQL string) |
| Type safety | Yes (compile-like errors) | No (runtime errors on bad SQL) |
| Readability | Good for complex logic | Great for analysts / SQL-first teams |
| Performance | Identical — same Catalyst optimizer | Identical |

> **Key insight:** Choose based on preference. Both compile to the same physical plan.

In [ ]:
from pyspark.sql.functions import col

data = [
    (1,  "Alice",   "Engineering", "RO", 95000),
    (2,  "Bob",     "Marketing",   "RO", 72000),
    (3,  "Carol",   "Engineering", "UK", 110000),
    (4,  "Dave",    "Sales",        "DE", 58000),
    (5,  "Eve",     "Marketing",    "RO", 81000),
    (6,  "Frank",   "Engineering",  "DE", 88000),
    (7,  "George",  "Sales",        "UK", 61000),
    (8,  "Helen",   "Engineering",  "RO", 102000),
]
df = spark.createDataFrame(data, ["id", "name", "dept", "country", "salary"])
df.show()

## 2. createOrReplaceTempView()

Registers the DataFrame as a **temporary view** — visible only within the current SparkSession.

- Lives for the duration of the session
- Replaced (not errored) if a view with the same name already exists
- Scoped to the current session — other sessions cannot see it

In [ ]:
# Register as a temp view
df.createOrReplaceTempView("employees")

# Now query it with SQL
spark.sql("SELECT * FROM employees").show()

## 3. Running SQL Queries with spark.sql()

In [ ]:
# SELECT with WHERE
spark.sql("""
    SELECT name, dept, salary
    FROM employees
    WHERE salary > 80000
    ORDER BY salary DESC
""").show()

# GROUP BY + aggregation
spark.sql("""
    SELECT dept,
           COUNT(*)          AS headcount,
           AVG(salary)       AS avg_salary,
           MAX(salary)       AS max_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""").show()

# CASE WHEN
spark.sql("""
    SELECT name, salary,
           CASE
               WHEN salary >= 100000 THEN 'Band A'
               WHEN salary >= 80000  THEN 'Band B'
               ELSE                       'Band C'
           END AS band
    FROM employees
""").show()

## 4. Global Temp Views

| | Temp View | Global Temp View |
|---|---|---|
| Created with | `createOrReplaceTempView()` | `createOrReplaceGlobalTempView()` |
| Scope | Current session only | All sessions in the same application |
| SQL namespace | `table_name` | **`global_temp.table_name`** |
| Lifetime | Until session ends | Until application ends |

> **Exam tip:** Global temp views must be prefixed with `global_temp.` in queries.

In [ ]:
# Register as a global temp view
df.createOrReplaceGlobalTempView("global_employees")

# Must use the global_temp. prefix
spark.sql("SELECT name, salary FROM global_temp.global_employees WHERE country = 'RO'").show()

## 5. Mixing SQL and DataFrame API

`spark.sql()` returns a **DataFrame** — you can chain any DataFrame operation on the result.

In [ ]:
# SQL result is a DataFrame — chain DataFrame operations
sql_result = spark.sql("SELECT * FROM employees WHERE dept = 'Engineering'")

# Now use the DataFrame API on it
sql_result.withColumn("salary_eur", col("salary") * 0.92) \
           .select("name", "salary", "salary_eur") \
           .show()

# Register the SQL result as another temp view for further SQL
sql_result.createOrReplaceTempView("engineers")
spark.sql("SELECT name, salary FROM engineers ORDER BY salary DESC").show()

## 6. Managing Views with spark.catalog

In [ ]:
# List all tables/views in the current session
spark.catalog.listTables()

# Check if a specific table exists
print(spark.catalog.tableExists("employees"))         # True
print(spark.catalog.tableExists("nonexistent_table")) # False

# Drop a temp view when no longer needed
spark.catalog.dropTempView("engineers")
print(spark.catalog.tableExists("engineers"))         # False after drop

## 7. SQL JOINs

SQL joins work exactly as in standard SQL — same syntax, same semantics.

In [ ]:
dept_data = [
    ("Engineering", "Tech",  True),
    ("Marketing",   "Growth", True),
    ("Sales",       "Revenue", False),
    ("Finance",     "Corp",   True),  # no employees in this dept
]
dept_df = spark.createDataFrame(dept_data, ["dept", "division", "hiring"])
dept_df.createOrReplaceTempView("departments")

# INNER JOIN — only matching rows
spark.sql("""
    SELECT e.name, e.salary, d.division, d.hiring
    FROM employees e
    INNER JOIN departments d ON e.dept = d.dept
    ORDER BY e.salary DESC
""").show()

# LEFT JOIN — all employees, even if dept has no match
spark.sql("""
    SELECT e.name, e.dept, d.division
    FROM employees e
    LEFT JOIN departments d ON e.dept = d.dept
""").show()

---

## Day 6 Checklist

- [ ] Registered a DataFrame as a temp view with `createOrReplaceTempView()`
- [ ] Ran SELECT, WHERE, GROUP BY, ORDER BY, CASE WHEN with `spark.sql()`
- [ ] Know the difference between temp view (session-scoped) and global temp view (app-scoped)
- [ ] Know that global temp views must use the `global_temp.` prefix
- [ ] Used `spark.sql()` result as a DataFrame and chained operations
- [ ] Used `spark.catalog` to list and drop views
- [ ] Ran SQL JOINs

**Next:** Day 7 — Reading & Writing Data (CSV, JSON, Parquet, schema, write modes)